# MoWM — Extract NV-Embed-v2 Embeddings for Zero-Shot VSA Projection

This notebook extracts embeddings from **nvidia/NV-Embed-v2** (4096D, #1 MTEB) and saves them as a domain codebook for MoWM's `SequenceEncoder.encode_embedded_tokens()`.

**Why NV-Embed-v2 over Qwen hidden states:**
- Purpose-built embedding model (not hidden-state extraction from a generative model)
- 4096D output — richer than Qwen2.5-7B's 3584D hidden states
- #1 on MTEB leaderboard — state-of-the-art retrieval, classification, clustering
- Instruction-tuned — supports task-prefixed queries for domain-specific encoding
- Bidirectional attention — captures full context (generative models are causal-only)

**Hardware:** Colab A100 (96GB VRAM) — runs NV-Embed-v2 at full precision

**Process:**
1. Load NV-Embed-v2 on Colab A100
2. Extract embeddings from training data with task instructions
3. Group by domain, compute centroids
4. Save as `.npz` codebook for MoWM

**Output:** `mowm_nvembed_codebook.npz` with:
- `embeddings`: (N, 4096) float32 array of all embeddings
- `texts`: list of source texts
- `domains`: list of domain labels per embedding
- `centroids`: (n_domains, 4096) domain centroid embeddings
- `domain_names`: list of domain names matching centroid rows

## 1. Setup & Install

In [ ]:
# Pin transformers to compatible version for NV-Embed-v2
!pip install -q "transformers==4.42.4" torch accelerate sentencepiece
!pip install -q datasets einops

In [ ]:
import json
import os
from pathlib import Path

import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2. Load NV-Embed-v2

nvidia/NV-Embed-v2 outputs 4096D embeddings. On A100 96GB it runs at full bf16 precision.
The model uses a latent attention layer on top of a Mistral-7B backbone — bidirectional attention
means it sees the full context (unlike causal LLMs).

In [ ]:
MODEL_ID = "nvidia/NV-Embed-v2"
LATENT_DIM = 4096  # NV-Embed-v2 output dimension

print(f"Loading {MODEL_ID}...")

# NV-Embed-v2 uses a custom NVEmbedModel class via trust_remote_code.
# It exposes an .encode() method directly — no need for AutoModelForCausalLM.
# If the model class is missing 'all_tied_weights_keys', we patch it.
from transformers import AutoModel

model = AutoModel.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

# Patch for newer transformers if needed
if not hasattr(model, "all_tied_weights_keys"):
    model.all_tied_weights_keys = []

model.eval()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

# Add pad token if missing (NV-Embed-v2 sometimes doesn't set it)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded on {next(model.parameters()).device}")
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## 3. Embedding Extraction Function

NV-Embed-v2 supports **instruction-prefixed** encoding. Each query is prefixed with a task
instruction that tells the model what kind of embedding to produce:
- `"Instruct: Classify this text into a scientific domain\nQuery: ..."` for domain anchors
- `"Instruct: Represent this text for semantic retrieval\nQuery: ..."` for general text

This gives better domain separation than generic mean-pooling.

In [ ]:
# NV-Embed-v2 task instructions
TASK_CLASSIFY = "Instruct: Classify this text by its knowledge domain\nQuery: "
TASK_RETRIEVE = "Instruct: Represent this text for semantic retrieval\nQuery: "
TASK_CLUSTER = "Instruct: Identify the core concept in this text\nQuery: "

# Check if model has native .encode() method (NVEmbedModel exposes it)
HAS_NATIVE_ENCODE = hasattr(model, "encode")
print(f"Native .encode() available: {HAS_NATIVE_ENCODE}")


@torch.no_grad()
def extract_embedding(
    text: str,
    task_instruction: str = TASK_RETRIEVE,
    max_length: int = 512,
) -> np.ndarray:
    """Extract embedding from NV-Embed-v2.

    Uses native .encode() if available, otherwise falls back to
    hidden state extraction with mean pooling.

    Args:
        text: Input text string.
        task_instruction: Instruction prefix for the embedding task.
        max_length: Maximum token length.

    Returns:
        (LATENT_DIM,) float32 numpy array, L2-normalized.
    """
    full_text = task_instruction + text

    if HAS_NATIVE_ENCODE:
        # Native NV-Embed-v2 encode — handles pooling internally
        emb = model.encode([full_text], max_length=max_length)
        if isinstance(emb, torch.Tensor):
            embedding = emb[0].float().cpu().numpy()
        else:
            embedding = np.array(emb[0], dtype=np.float32)
    else:
        # Fallback: manual hidden state extraction
        inputs = tokenizer(
            full_text,
            return_tensors="pt",
            truncation=True,
            max_length=max_length,
            padding=False,
        ).to(next(model.parameters()).device)

        outputs = model(**inputs, output_hidden_states=True, return_dict=True)
        last_hidden = outputs.hidden_states[-1]
        attention_mask = inputs["attention_mask"].unsqueeze(-1)
        masked = last_hidden * attention_mask
        pooled = masked.sum(dim=1) / attention_mask.sum(dim=1).clamp(min=1)
        embedding = pooled[0].float().cpu().numpy()

    # L2 normalize
    norm = np.linalg.norm(embedding)
    if norm > 0:
        embedding = embedding / norm
    return embedding.astype(np.float32)


@torch.no_grad()
def extract_embeddings_batch(
    texts: list[str],
    task_instruction: str = TASK_RETRIEVE,
    batch_size: int = 16,
    max_length: int = 512,
) -> np.ndarray:
    """Extract embeddings for a batch of texts.

    Args:
        texts: List of input strings.
        task_instruction: Instruction prefix for all texts.
        batch_size: Texts per batch (A100 handles 16+).
        max_length: Maximum token length.

    Returns:
        (N, LATENT_DIM) float32 numpy array, L2-normalized.
    """
    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch_texts = [task_instruction + t for t in texts[i:i + batch_size]]

        if HAS_NATIVE_ENCODE:
            emb = model.encode(batch_texts, max_length=max_length)
            if isinstance(emb, torch.Tensor):
                embeddings = emb.float().cpu().numpy()
            else:
                embeddings = np.array(emb, dtype=np.float32)
        else:
            inputs = tokenizer(
                batch_texts,
                return_tensors="pt",
                truncation=True,
                max_length=max_length,
                padding=True,
            ).to(next(model.parameters()).device)

            outputs = model(**inputs, output_hidden_states=True, return_dict=True)
            last_hidden = outputs.hidden_states[-1]
            attention_mask = inputs["attention_mask"].unsqueeze(-1)
            masked = last_hidden * attention_mask
            pooled = masked.sum(dim=1) / attention_mask.sum(dim=1).clamp(min=1)
            embeddings = pooled.float().cpu().numpy()

        # L2 normalize each
        norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
        norms = np.where(norms == 0, 1.0, norms)
        embeddings = (embeddings / norms).astype(np.float32)
        all_embeddings.append(embeddings)

        if (i // batch_size) % 25 == 0:
            print(f"  Processed {i + len(batch_texts)}/{len(texts)} texts")

    return np.concatenate(all_embeddings, axis=0)


# Quick test
test_emb = extract_embedding("Force equals mass times acceleration")
print(f"Embedding shape: {test_emb.shape}, norm: {np.linalg.norm(test_emb):.4f}")
assert test_emb.shape[0] > 0, "Empty embedding!"
# Update LATENT_DIM if model outputs different size
actual_dim = test_emb.shape[0]
if actual_dim != LATENT_DIM:
    print(f"NOTE: Actual dim {actual_dim} differs from expected {LATENT_DIM}, updating.")
    LATENT_DIM = actual_dim

# Semantic test
emb_phys = extract_embedding("Newton's law of motion", TASK_CLASSIFY)
emb_music = extract_embedding("Bach fugue in D minor", TASK_CLASSIFY)
emb_phys2 = extract_embedding("Gravitational acceleration", TASK_CLASSIFY)
print(f"\nphysics-physics sim: {float(emb_phys @ emb_phys2):.3f}")
print(f"physics-music sim:   {float(emb_phys @ emb_music):.3f}")

## 4. Mount Google Drive & Set All Data Paths

Upload ALL your training data to Drive. The notebook processes everything:
- `E:\RAW_TEXTS\realms\` — 5,998 files across 35 realms (history, physics, identity, finance, etc.)
- `D:\grillcheese_training_data\` — SVC, grammar, planning, chat templates
- Domain anchor sentences (built-in, 65 core concepts)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_BASE = "/content/drive/MyDrive/mowm_data"

# ── Primary data sources ──────────────────────────────────────────────
REALMS_PATH = f"{DRIVE_BASE}/realms"                      # E:\RAW_TEXTS\realms
WSJ_PATHS = [f"{DRIVE_BASE}/WSJ {q} Qtr 2025"            # E:\RAW_TEXTS\WSJ *
             for q in ["1st", "2nd", "3rd", "4th"]]
CUBEMIND_TRAINING_PATH = f"{DRIVE_BASE}/cubemind_training" # E:\RAW_TEXTS\cubemind_training
PROCESSED_PATH = f"{DRIVE_BASE}/processed"                 # E:\RAW_TEXTS\processed

# ── D:\grillcheese_training_data (full directory) ─────────────────────
GC_DATA = f"{DRIVE_BASE}/grillcheese_training_data"
SVC_INSTRUCT_PATH = f"{GC_DATA}/instruct_svc_semantic.jsonl"
SVC_CONVO_PATH = f"{GC_DATA}/conversations_svc_semantic.jsonl"
GRAMMAR_PATH = f"{GC_DATA}/pre/combined_grammar.jsonl"
PLANS_PATH = f"{GC_DATA}/pre/learning-plans_batch_2025-07-26T06-41-56-841Z.jsonl"
FACTUAL_PATH = f"{GC_DATA}/factual"                        # 3954 files
GBOOKS_PATH = f"{GC_DATA}/gbooks"                          # 6546 files
KNOWLEDGE_PATH = f"{GC_DATA}/knowledgetxt"                 # 1114 books
TEMPORAL_PATH = f"{GC_DATA}/temporal"                      # 303 historical
PRE_PATH = f"{GC_DATA}/pre"                                # 32 affect/chat files
JSONL_PATH = f"{GC_DATA}/jsonl"                            # OpenThoughts + Wikibooks
UNIFIED_PATH = f"{GC_DATA}/unified"                        # C4 multilingual + news

OUTPUT_PATH = f"{DRIVE_BASE}/mowm_nvembed_codebook.npz"

# Map realm directories to MoWM domains
REALM_TO_DOMAIN = {
    "quantum_mechanics": "physics", "general_science": "physics",
    "electronic": "physics", "transport": "physics",
    "math": "logic", "cognition": "logic", "philosophy": "logic",
    "finance": "economics", "business_management": "economics",
    "marketing": "economics", "data_management": "economics", "startup": "economics",
    "psychology": "social", "self_improvement": "social",
    "language": "language", "instruction": "language",
    "legal": "logic", "governance": "logic",
    "health": "bio", "neuroscience": "bio",
    "history": "historical", "temporal_timelines_events": "historical",
    "my_identity": "identity", "conversation": "identity",
    "thinking": "causal", "IT": "security",
    "software_engineering": "language", "ml": "ensemble",
    "labeling": "ensemble", "QA": "language",
    "verified_facts": "general", "general_mainstream": "general",
    "gutemberg_books_unclassified": "general", "factual_books": "general",
}

# Quick availability check
print(f"Drive base: {DRIVE_BASE}\n")
sources = {
    "Realms": REALMS_PATH, "WSJ Q1": WSJ_PATHS[0],
    "cubemind_training": CUBEMIND_TRAINING_PATH,
    "GC factual": FACTUAL_PATH, "GC gbooks": GBOOKS_PATH,
    "GC knowledge": KNOWLEDGE_PATH, "GC temporal": TEMPORAL_PATH,
    "GC pre": PRE_PATH, "GC jsonl": JSONL_PATH,
    "GC unified": UNIFIED_PATH,
    "SVC instruct": SVC_INSTRUCT_PATH, "SVC convo": SVC_CONVO_PATH,
}
for label, p in sources.items():
    ok = os.path.exists(p)
    print(f"  {'OK' if ok else 'MISSING'}: {label}")

## 5. Load ALL Training Data by Domain

Processes:
1. Domain anchor sentences (65 core concepts — highest priority)
2. All 35 realm directories from E:\RAW_TEXTS\realms
3. SVC instruction + conversation data (with domain labels)
4. Grammar, planning, and chat template data
5. Temporal timeline events (historical JSONs)

Each text chunk gets a domain label. Target: 50k-100k embeddings for comprehensive coverage.

In [ ]:
def load_jsonl(path: str, max_records: int = 5000) -> list[dict]:
    """Load a JSONL file, return list of dicts."""
    records = []
    if not os.path.exists(path):
        print(f"  SKIP (not found): {path}")
        return records
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if len(records) >= max_records:
                break
            line = line.strip()
            if line:
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError:
                    continue
    print(f"  Loaded {len(records)} records from {os.path.basename(path)}")
    return records


def extract_text(record: dict) -> str:
    """Extract text from a training record (handles various formats)."""
    # Try common field names
    for key in ["text", "instruction", "input", "content", "question", "prompt"]:
        if key in record and record[key]:
            val = record[key]
            if isinstance(val, str) and len(val) > 10:
                return val[:1000]  # cap at 1000 chars
    # Fallback: concatenate all string values
    parts = [str(v) for v in record.values() if isinstance(v, str) and len(str(v)) > 5]
    return " ".join(parts)[:1000] if parts else ""

In [ ]:
# Domain-specific axiom sentences for anchor embeddings
# These are the "core concepts" that define each domain in embedding space
DOMAIN_ANCHORS: dict[str, list[str]] = {
    "physics": [
        "Force equals mass times acceleration, F = m * a",
        "Energy is conserved in isolated systems",
        "The velocity of an object changes when a net force acts on it",
        "Gravitational force between two masses is proportional to their product",
        "Kinetic energy equals one half mass times velocity squared",
        "Pressure equals force divided by area",
        "The wave equation relates frequency, wavelength, and velocity",
        "Ohm's law states voltage equals current times resistance",
    ],
    "logic": [
        "If A implies B and A is true, then B must be true (modus ponens)",
        "A proposition cannot be both true and false at the same time",
        "De Morgan's law: NOT(A AND B) equals NOT A OR NOT B",
        "If an obligation exists and no exception applies, the action is required",
        "A valid deductive argument preserves truth from premises to conclusion",
    ],
    "economics": [
        "Supply and demand determine market equilibrium price",
        "Nash equilibrium is where no player benefits from changing strategy alone",
        "GDP equals consumption plus investment plus government spending plus net exports",
        "The Fisher equation relates nominal interest rate to real rate and inflation",
        "Marginal utility decreases as consumption increases",
    ],
    "social": [
        "PageRank assigns importance based on incoming links from important pages",
        "Network diffusion models how information spreads through social connections",
        "Preferential attachment means popular nodes attract more connections",
        "The SIR model tracks susceptible, infected, and recovered populations",
    ],
    "causal": [
        "Bayes theorem computes posterior probability from prior and likelihood",
        "Causal intervention do(X) removes incoming edges to X in the causal graph",
        "Counterfactual reasoning asks what would have happened if X were different",
        "Granger causality tests whether past values of X improve prediction of Y",
    ],
    "atomic": [
        "The Schrodinger equation describes quantum state evolution over time",
        "Heisenberg uncertainty principle limits simultaneous measurement of position and momentum",
        "Electrons occupy discrete energy levels in atoms",
        "The Lennard-Jones potential models intermolecular attraction and repulsion",
    ],
    "bio": [
        "Hardy-Weinberg equilibrium describes allele frequencies in stable populations",
        "Hamilton's rule for altruism: relatedness times benefit exceeds cost",
        "DNA encodes genetic information through nucleotide base pairs A-T and G-C",
        "Natural selection favors traits that increase reproductive fitness",
        "Michaelis-Menten kinetics describe enzyme reaction rates",
    ],
    "music": [
        "Harmonic series frequencies are integer multiples of the fundamental",
        "Equal temperament divides the octave into twelve equal semitones",
        "Consonance correlates with simple frequency ratios between notes",
        "Fourier decomposition represents any signal as a sum of sinusoids",
    ],
    "identity": [
        "Personal identity is shaped by core values, beliefs, and experiences",
        "Emotional baseline varies across eight primary emotions in Plutchik's model",
        "Family bonds and cultural heritage form the foundation of personal identity",
        "Habits strengthen through repetition following an exponential learning curve",
    ],
    "colors": [
        "Visible light wavelength determines perceived color from violet to red",
        "RGB additive color mixing combines red green and blue light",
        "Complementary colors are opposite on the color wheel",
        "Color temperature in Kelvin describes warm to cool light appearance",
    ],
    "language": [
        "Sentence structure follows constituency grammar with noun and verb phrases",
        "Word meaning is determined by distributional context in large corpora",
        "Morphemes combine to form words through inflection and derivation",
        "Zipf's law relates word frequency to its rank in the distribution",
    ],
    "historical": [
        "Historical events form causal chains where earlier events enable later ones",
        "Civilizations follow cycles of rise, peak, decline, and renewal",
        "Technology adoption follows an S-curve from early adopters to mainstream",
        "Counterfactual history explores what might have happened differently",
    ],
    "ensemble": [
        "Population behavior emerges from aggregating many individual decisions",
        "Mean-field approximation replaces individual interactions with average effect",
        "Phase transitions occur when system parameters cross critical thresholds",
        "Wisdom of crowds: aggregate estimates converge to true values",
    ],
    "linguistics": [
        "Context-free grammars generate sentences through production rules",
        "Dependency grammar represents syntactic structure through head-dependent relations",
        "Binding theory constrains the interpretation of pronouns and anaphors",
    ],
}

print(f"Domain anchors: {len(DOMAIN_ANCHORS)} domains")
total_anchors = sum(len(v) for v in DOMAIN_ANCHORS.values())
print(f"Total anchor sentences: {total_anchors}")

In [ ]:
# ── Load ALL data sources ──────────────────────────────────────────────

all_texts: list[str] = []
all_domains: list[str] = []
CHUNK_SIZE = 10  # lines per chunk for .txt files
MAX_PER_REALM = 2000  # cap per realm directory to keep balanced
WSJ_CHUNK_SIZE = 20  # WSJ articles are longer, use bigger chunks
MAX_PER_WSJ_QTR = 3000  # WSJ has dense content

# ── 1. Domain anchors (highest priority) ──────────────────────────────
for domain, sentences in DOMAIN_ANCHORS.items():
    for sent in sentences:
        all_texts.append(sent)
        all_domains.append(domain)
print(f"1. Anchors: {len(all_texts)} texts")

# ── 2. ALL realm directories ──────────────────────────────────────────
if os.path.exists(REALMS_PATH):
    realm_dirs = sorted([d for d in os.listdir(REALMS_PATH)
                         if os.path.isdir(os.path.join(REALMS_PATH, d))])
    for realm_name in realm_dirs:
        domain = REALM_TO_DOMAIN.get(realm_name, "general")
        realm_path = Path(os.path.join(REALMS_PATH, realm_name))
        count_before = len(all_texts)
        realm_count = 0

        for fpath in sorted(realm_path.rglob("*")):
            if realm_count >= MAX_PER_REALM:
                break
            if not fpath.is_file():
                continue

            try:
                if fpath.suffix == ".txt":
                    text = fpath.read_text(encoding="utf-8", errors="replace")
                    lines = text.strip().split("\n")
                    for i in range(0, len(lines), CHUNK_SIZE):
                        if realm_count >= MAX_PER_REALM:
                            break
                        chunk = "\n".join(lines[i:i + CHUNK_SIZE]).strip()
                        if len(chunk) > 20:
                            all_texts.append(chunk[:1000])
                            all_domains.append(domain)
                            realm_count += 1

                elif fpath.suffix == ".jsonl":
                    with open(fpath, "r", encoding="utf-8") as f:
                        for line in f:
                            if realm_count >= MAX_PER_REALM:
                                break
                            line = line.strip()
                            if not line:
                                continue
                            try:
                                rec = json.loads(line)
                                text = extract_text(rec)
                                if text and len(text) > 20:
                                    all_texts.append(text[:1000])
                                    all_domains.append(domain)
                                    realm_count += 1
                            except json.JSONDecodeError:
                                continue

                elif fpath.suffix == ".json":
                    with open(fpath, "r", encoding="utf-8") as f:
                        data = json.load(f)
                    if isinstance(data, list):
                        for item in data[:MAX_PER_REALM - realm_count]:
                            if isinstance(item, dict):
                                text = extract_text(item)
                            else:
                                text = str(item)
                            if text and len(text) > 20:
                                all_texts.append(text[:1000])
                                all_domains.append(domain)
                                realm_count += 1
                    elif isinstance(data, dict):
                        text = extract_text(data)
                        if text and len(text) > 20:
                            all_texts.append(text[:1000])
                            all_domains.append(domain)
                            realm_count += 1
            except Exception:
                continue

        added = len(all_texts) - count_before
        if added > 0:
            print(f"   {realm_name:40s} -> {domain:12s} +{added}")

print(f"\n2. Realms total: {len(all_texts)} texts")

# ── 3. WSJ 2025 — all 4 quarters (economics/historical) ──────────────
wsj_count = 0
for wsj_path in WSJ_PATHS:
    if not os.path.exists(wsj_path):
        continue
    qtr_count = 0
    for fpath in sorted(Path(wsj_path).rglob("*.txt")):
        if qtr_count >= MAX_PER_WSJ_QTR:
            break
        try:
            text = fpath.read_text(encoding="utf-8", errors="replace")
            lines = text.strip().split("\n")
            # Extract date from filename for temporal context
            fname = fpath.stem  # e.g. wallstreetjournal_20250201_...
            for i in range(0, len(lines), WSJ_CHUNK_SIZE):
                if qtr_count >= MAX_PER_WSJ_QTR:
                    break
                chunk = "\n".join(lines[i:i + WSJ_CHUNK_SIZE]).strip()
                if len(chunk) > 30:
                    all_texts.append(chunk[:1000])
                    # WSJ covers economics, finance, politics, tech
                    all_domains.append("economics")
                    qtr_count += 1
        except Exception:
            continue
    wsj_count += qtr_count
    print(f"   WSJ {os.path.basename(wsj_path):25s} +{qtr_count}")

print(f"\n3. WSJ total: +{wsj_count} chunks")

# ── 4. cubemind_training (realm pairs) ────────────────────────────────
cm_count = 0
if os.path.exists(CUBEMIND_TRAINING_PATH):
    for fpath in sorted(Path(CUBEMIND_TRAINING_PATH).rglob("*.jsonl")):
        records = load_jsonl(str(fpath), max_records=2000)
        for rec in records:
            text = extract_text(rec)
            domain = rec.get("realm", rec.get("domain", "general"))
            if isinstance(domain, str):
                domain = domain.lower().split(".")[0]
            if text and len(text) > 20:
                all_texts.append(text[:1000])
                all_domains.append(str(domain))
                cm_count += 1
    print(f"4. cubemind_training: +{cm_count}")

# ── 5. processed data ─────────────────────────────────────────────────
proc_count = 0
if os.path.exists(PROCESSED_PATH):
    for fpath in sorted(Path(PROCESSED_PATH).rglob("*.jsonl")):
        records = load_jsonl(str(fpath), max_records=2000)
        for rec in records:
            text = extract_text(rec)
            if text and len(text) > 20:
                all_texts.append(text[:1000])
                all_domains.append("general")
                proc_count += 1
    print(f"5. processed: +{proc_count}")

# ── 6. SVC instruction data ───────────────────────────────────────────
svc_instruct = load_jsonl(SVC_INSTRUCT_PATH, max_records=5000)
for rec in svc_instruct:
    text = extract_text(rec)
    domain = rec.get("domain", rec.get("category", "general"))
    if isinstance(domain, str):
        domain = domain.lower().split(".")[0]
    if text and len(text) > 20:
        all_texts.append(text[:1000])
        all_domains.append(str(domain))

# ── 7. SVC conversation data ──────────────────────────────────────────
svc_convo = load_jsonl(SVC_CONVO_PATH, max_records=3000)
for rec in svc_convo:
    text = extract_text(rec)
    domain = rec.get("domain", rec.get("category", "general"))
    if isinstance(domain, str):
        domain = domain.lower().split(".")[0]
    if text and len(text) > 20:
        all_texts.append(text[:1000])
        all_domains.append(str(domain))

# ── 8. Grammar data ───────────────────────────────────────────────────
grammar = load_jsonl(GRAMMAR_PATH, max_records=2000)
for rec in grammar:
    text = extract_text(rec)
    if text and len(text) > 20:
        all_texts.append(text[:1000])
        all_domains.append("language")

# ── 9. Planning data ──────────────────────────────────────────────────
plans = load_jsonl(PLANS_PATH, max_records=1000)
for rec in plans:
    text = extract_text(rec)
    if text and len(text) > 20:
        all_texts.append(text[:1000])
        all_domains.append("general")

# ── Summary ───────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"TOTAL texts to embed: {len(all_texts)}")
print(f"\nDomain distribution:")
from collections import Counter
domain_counts = Counter(all_domains)
for domain, count in domain_counts.most_common(30):
    print(f"  {domain:20s} {count:>6d}")
print(f"\n  {'TOTAL':20s} {len(all_texts):>6d}")
est_gb = len(all_texts) * LATENT_DIM * 4 / 1e9
print(f"\nEstimated codebook size: {est_gb:.1f} GB (embeddings only)")

In [ ]:
# ── Additional: D:\grillcheese_training_data subdirectories ───────────

def load_txt_chunks(directory, domain, max_chunks=2000, chunk_size=10):
    """Load .txt files from a directory, chunked by lines."""
    count = 0
    if not os.path.exists(directory):
        return count
    for fpath in sorted(Path(directory).rglob("*.txt")):
        if count >= max_chunks:
            break
        try:
            text = fpath.read_text(encoding="utf-8", errors="replace")
            lines = text.strip().split("\n")
            for i in range(0, len(lines), chunk_size):
                if count >= max_chunks:
                    break
                chunk = "\n".join(lines[i:i + chunk_size]).strip()
                if len(chunk) > 20:
                    all_texts.append(chunk[:1000])
                    all_domains.append(domain)
                    count += 1
        except Exception:
            continue
    return count

# Factual texts (3954 files — multi-domain factual content)
n = load_txt_chunks(FACTUAL_PATH, "general", max_chunks=3000)
print(f"GC factual: +{n}")

# Google Books (6546 files — broad knowledge)
n = load_txt_chunks(GBOOKS_PATH, "general", max_chunks=3000)
print(f"GC gbooks: +{n}")

# Knowledge texts (1114 books — Oxford handbooks, history, science)
n = load_txt_chunks(KNOWLEDGE_PATH, "general", max_chunks=3000)
print(f"GC knowledge: +{n}")

# Temporal / historical (303 files)
n = load_txt_chunks(TEMPORAL_PATH, "historical", max_chunks=2000)
print(f"GC temporal: +{n}")

# Pre-training affect/chat data (32 files)
pre_count = 0
if os.path.exists(PRE_PATH):
    for fpath in sorted(Path(PRE_PATH).rglob("*.jsonl")):
        records = load_jsonl(str(fpath), max_records=500)
        for rec in records:
            text = extract_text(rec)
            if text and len(text) > 20:
                all_texts.append(text[:1000])
                all_domains.append("identity")
                pre_count += 1
print(f"GC pre (affect/chat): +{pre_count}")

# OpenThoughts-114k + Wikibooks (JSONL — sample)
jsonl_count = 0
if os.path.exists(JSONL_PATH):
    for fpath in sorted(Path(JSONL_PATH).glob("*.jsonl")):
        records = load_jsonl(str(fpath), max_records=3000)
        for rec in records:
            text = extract_text(rec)
            if text and len(text) > 20:
                all_texts.append(text[:1000])
                all_domains.append("general")
                jsonl_count += 1
print(f"GC jsonl (OpenThoughts/Wikibooks): +{jsonl_count}")

# C4 multilingual + real news (sample — these are huge)
unified_count = 0
if os.path.exists(UNIFIED_PATH):
    for fpath in sorted(Path(UNIFIED_PATH).glob("*.jsonl")):
        records = load_jsonl(str(fpath), max_records=2000)
        for rec in records:
            text = extract_text(rec)
            if text and len(text) > 20:
                all_texts.append(text[:1000])
                # C4 realnews -> economics/general
                if "realnews" in fpath.stem:
                    all_domains.append("economics")
                elif "multilingual" in fpath.stem:
                    all_domains.append("language")
                else:
                    all_domains.append("general")
                unified_count += 1
print(f"GC unified (C4/news): +{unified_count}")

# ── Final summary ─────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"GRAND TOTAL texts to embed: {len(all_texts)}")
print(f"\nDomain distribution:")
domain_counts = Counter(all_domains)
for domain, count in domain_counts.most_common(30):
    print(f"  {domain:20s} {count:>6d}")
print(f"\n  {'TOTAL':20s} {len(all_texts):>6d}")
est_gb = len(all_texts) * LATENT_DIM * 4 / 1e9
est_hours = len(all_texts) / 16 * 0.3 / 3600
print(f"\nEstimated codebook: {est_gb:.1f} GB | Extraction time: ~{est_hours:.1f}h on A100")

## 6. Extract Embeddings

In [ ]:
print(f"Extracting {len(all_texts)} embeddings with NV-Embed-v2...")
print(f"Batch size: 16 (A100 96GB), max_length: 512")
est_minutes = len(all_texts) / 16 * 0.3 / 60
print(f"Estimated time: ~{est_minutes:.0f} minutes on A100")
print(f"  ({est_minutes/60:.1f} hours)\n")

# Separate anchors (TASK_CLASSIFY) from general text (TASK_RETRIEVE)
n_anchors = total_anchors
anchor_texts = all_texts[:n_anchors]
general_texts = all_texts[n_anchors:]

print(f"Phase 1: {len(anchor_texts)} domain anchors with TASK_CLASSIFY...")
anchor_embeddings = extract_embeddings_batch(
    anchor_texts,
    task_instruction=TASK_CLASSIFY,
    batch_size=16,
    max_length=512,
)

print(f"\nPhase 2: {len(general_texts)} general texts with TASK_RETRIEVE...")
general_embeddings = extract_embeddings_batch(
    general_texts,
    task_instruction=TASK_RETRIEVE,
    batch_size=16,
    max_length=512,
)

embeddings = np.concatenate([anchor_embeddings, general_embeddings], axis=0)

print(f"\nDone! Embeddings shape: {embeddings.shape}")
print(f"Dtype: {embeddings.dtype}")
print(f"Size in memory: {embeddings.nbytes / 1e9:.2f} GB")
print(f"Mean norm: {np.linalg.norm(embeddings, axis=1).mean():.4f}")
print(f"VRAM peak: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

## 7. Compute Domain Centroids

In [ ]:
# Group embeddings by domain and compute centroids
domain_names_unique = sorted(set(all_domains))
centroids: dict[str, np.ndarray] = {}

print("Computing domain centroids...\n")
for domain in domain_names_unique:
    indices = [i for i, d in enumerate(all_domains) if d == domain]
    if not indices:
        continue
    domain_embs = embeddings[indices]
    centroid = domain_embs.mean(axis=0)
    # L2 normalize
    norm = np.linalg.norm(centroid)
    if norm > 0:
        centroid = centroid / norm
    centroids[domain] = centroid.astype(np.float32)
    print(f"  {domain}: {len(indices)} embeddings -> centroid norm {np.linalg.norm(centroid):.4f}")

print(f"\nTotal centroids: {len(centroids)}")

In [ ]:
# Verify: domain centroids should be distinguishable
print("Cross-domain cosine similarity (should be < 0.9 for different domains):\n")
mowm_domains = [
    "physics", "logic", "economics", "social", "causal",
    "atomic", "bio", "music", "identity", "colors",
    "language", "historical", "ensemble",
]
available_domains = [d for d in mowm_domains if d in centroids]

# Print similarity matrix header
header = "         " + " ".join(f"{d[:6]:>6}" for d in available_domains)
print(header)

for d1 in available_domains:
    row = f"{d1[:8]:>8} "
    for d2 in available_domains:
        sim = float(centroids[d1] @ centroids[d2])
        row += f"{sim:6.3f} "
    print(row)

## 8. Multilingual Verification

Verify that Qwen2.5's multilingual embeddings preserve cross-language similarity.

In [ ]:
# Test multilingual alignment
multilingual_pairs = [
    ("Force equals mass times acceleration", "La force est egale a la masse fois l'acceleration"),
    ("Force equals mass times acceleration", "力等于质量乘以加速度"),
    ("Energy is conserved", "L'energie est conservee"),
    ("Energy is conserved", "エネルギーは保存される"),
    ("Supply and demand", "Offre et demande"),
    ("Family is important", "La famille est importante"),
    ("Family is important", "家族は大切です"),
]

print("Multilingual cosine similarities (should be > 0.7):\n")
for en, other in multilingual_pairs:
    emb_en = extract_embedding(en)
    emb_other = extract_embedding(other)
    sim = float(emb_en @ emb_other)
    lang = "FR" if any(c in other for c in "éèêë") else (
        "ZH" if any('\u4e00' <= c <= '\u9fff' for c in other) else (
            "JA" if any('\u3040' <= c <= '\u30ff' for c in other) else "??"
        )
    )
    print(f"  EN-{lang} sim={sim:.3f}  |  '{en[:40]}' vs '{other[:40]}'")

## 9. Save Codebook

In [ ]:
# Precompute the orthogonal projection matrix P on Colab
# This avoids the expensive QR decomposition on your local machine
K_VSA, L_VSA = 80, 128
vsa_dim = K_VSA * L_VSA  # 10240

print(f"Computing orthogonal projection matrix P: ({vsa_dim}, {LATENT_DIM})...")
print("(This takes ~30s on Colab but saves minutes locally)")

rng = np.random.default_rng(42)
gaussian = rng.standard_normal((vsa_dim, LATENT_DIM))
P, _ = np.linalg.qr(gaussian)
P = P.astype(np.float32)
print(f"P shape: {P.shape}, dtype: {P.dtype}")

# Verify orthogonality: P^T @ P should be close to identity
PtP = P.T @ P
off_diag = np.abs(PtP - np.eye(LATENT_DIM)).max()
print(f"Orthogonality check (max off-diagonal): {off_diag:.2e} (should be < 1e-5)")

# Save codebook WITH projection matrix included
centroid_names = sorted(centroids.keys())
centroid_matrix = np.stack([centroids[d] for d in centroid_names], axis=0)

print(f"\nSaving codebook + projection matrix to {OUTPUT_PATH}...")
np.savez_compressed(
    OUTPUT_PATH,
    embeddings=embeddings,
    texts=np.array(all_texts, dtype=object),
    domains=np.array(all_domains, dtype=object),
    centroids=centroid_matrix,
    domain_names=np.array(centroid_names, dtype=object),
    model_id=MODEL_ID,
    latent_dim=LATENT_DIM,
    projection_matrix=P,  # <-- precomputed, load this locally!
    k_vsa=K_VSA,
    l_vsa=L_VSA,
)

file_size_mb = os.path.getsize(OUTPUT_PATH) / (1024 * 1024)
print(f"Saved! File size: {file_size_mb:.1f} MB")
print(f"  (projection matrix adds ~{P.nbytes / 1e6:.0f} MB)")

## 10. Verify: Load and Use with MoWM SequenceEncoder

This cell shows how to use the codebook with MoWM (run locally, not on Colab).

In [ ]:
# Example: how to use locally — NO QR computation needed!
print("""
# ── Usage with MoWM (run locally — instant, no heavy QR) ──────

from mowm.sequence_encoder import SequenceEncoder

# Option 1: Load from codebook (includes precomputed P matrix)
enc = SequenceEncoder.from_codebook("mowm_nvembed_codebook.npz")
# That's it! P is loaded from the .npz, no QR decomposition.

# Option 2: Load P separately
import numpy as np
cb = np.load("mowm_nvembed_codebook.npz", allow_pickle=True)
enc = SequenceEncoder(k=80, l=128, projection_matrix=cb["projection_matrix"])

# Project domain centroids
centroids = cb["centroids"]
domain_names = list(cb["domain_names"])
physics_vsa = enc._project_embedding(centroids[domain_names.index("physics")])

# Encode a sequence of NV-Embed-v2 embeddings
embeddings = cb["embeddings"]
seq_vec = enc.encode_embedded_tokens([embeddings[0], embeddings[1]])

# Zero-shot: similar concepts get similar VSA vectors
from cubemind.ops.block_codes import BlockCodes
bc = BlockCodes(k=80, l=128)
v_phys = enc._project_embedding(centroids[domain_names.index("physics")])
v_atom = enc._project_embedding(centroids[domain_names.index("atomic")])
print(f"Physics-Atomic VSA sim: {bc.similarity(v_phys, v_atom):.3f}")
""")

In [ ]:
print("Done! Download mowm_nvembed_codebook.npz from Google Drive.")
print(f"File: {OUTPUT_PATH}")
print(f"Size: {file_size_mb:.1f} MB")
print(f"Contains {embeddings.shape[0]} embeddings x {LATENT_DIM}D from {MODEL_ID}")
print(f"\nTo use locally:")
print(f"  SequenceEncoder(k=80, l=128, latent_dim={LATENT_DIM})")
print(f"  enc.encode_embedded_tokens([emb1, emb2, ...])  # zero-shot!")